# ChroniclingAmericaQA — Data Preprocessing

**Dataset:** [Bhawna/ChroniclingAmericaQA](https://huggingface.co/datasets/Bhawna/ChroniclingAmericaQA)

This notebook covers:
1. Loading the ChroniclingAmericaQA dataset
2. Exploratory data inspection (field overview, length statistics)
3. Preprocessing (whitespace normalization, OCR toggle, truncation)
4. Prompt formatting for instruction-tuned models (Phi-3.5 chat template)

---
## 1. Setup and Imports

In [ ]:
# Uncomment and run once to install dependencies
# !pip install datasets transformers
# !pip install pandas numpy tqdm
# !pip install evaluate rouge_score

In [ ]:
import os
import re
import json
import string
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import set_seed

warnings.filterwarnings("ignore")
print("Imports OK.")

---
## 2. Configuration

All hyperparameters live here. Adjust before running any downstream cell.

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────
DATASET_NAME = "Bhawna/ChroniclingAmericaQA"

# How many samples to use per split (None = use full split)
MAX_TRAIN_SAMPLES = 500
MAX_DEV_SAMPLES   = 100
MAX_TEST_SAMPLES  = 200

# Whether to use raw OCR text instead of cleaned context
USE_RAW_OCR = False        # toggle for OCR vs clean context experiments

# ── Sequence lengths ────────────────────────────────────────────────────────
MAX_INPUT_LENGTH = 1024    # approximate token budget for context + prompt
MAX_TARGET_LENGTH = 64     # max answer length in tokens

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED       = 42
OUTPUT_DIR = Path("./outputs/chronicling_qa")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print("Configuration loaded.")
print(f"  Dataset    : {DATASET_NAME}")
print(f"  USE_RAW_OCR: {USE_RAW_OCR}")
print(f"  Output dir : {OUTPUT_DIR.resolve()}")

---
## 3. Load Dataset

We load ChroniclingAmericaQA from the Hugging Face Hub. If the identifier needs updating, edit `DATASET_NAME` in Section 2.

In [36]:
# TODO: If the dataset requires authentication, run:
#   from huggingface_hub import login; login(token="hf_YOUR_TOKEN_HERE")

print(f"Loading dataset: {DATASET_NAME} ...")

try:
    raw_datasets = load_dataset(DATASET_NAME)
except Exception as e:
    # TODO: If the dataset isn't on HF Hub, load from local files instead:
    # raw_datasets = load_dataset("json", data_files={
    #     "train": "path/to/train.json",
    #     "validation": "path/to/dev.json",
    #     "test": "path/to/test.json",
    # })
    raise RuntimeError(
        f"Could not load dataset '{DATASET_NAME}'. "
        "Update DATASET_NAME or load from local files (see TODO above)."
    ) from e

print("\nDataset splits:")
for split_name, split_data in raw_datasets.items():
    print(f"  {split_name:12s}: {len(split_data):>6,} examples")

print("\nColumn names:", raw_datasets[list(raw_datasets.keys())[0]].column_names)

Loading dataset: Bhawna/ChroniclingAmericaQA ...

Dataset splits:
  train       : 439,302 examples
  validation  : 24,111 examples
  test        : 24,084 examples

Column names: ['query_id', 'question', 'answer', 'org_answer', 'para_id', 'context', 'raw_ocr', 'publication_date', 'trans_que', 'trans_ans', 'url']


In [37]:
# Map split names to standard train / dev / test
# TODO: Adjust these keys if the dataset uses different split names
SPLIT_TRAIN = "train"
SPLIT_DEV   = "validation"   # may be 'dev' in some versions
SPLIT_TEST  = "test"

train_ds = raw_datasets[SPLIT_TRAIN]
dev_ds   = raw_datasets[SPLIT_DEV]
test_ds  = raw_datasets[SPLIT_TEST]

print(f"Train : {len(train_ds):,}")
print(f"Dev   : {len(dev_ds):,}")
print(f"Test  : {len(test_ds):,}")

Train : 439,302
Dev   : 24,111
Test  : 24,084


---
## 4. Exploratory Data Inspection

Understand the data schema, typical answer lengths, and how much OCR noise is present.

In [38]:
# Print a few raw examples
NUM_EXAMPLES_TO_SHOW = 3

for i, example in enumerate(train_ds.select(range(NUM_EXAMPLES_TO_SHOW))):
    print(f"{'='*70}")
    print(f"Example {i+1}")
    print(f"{'='*70}")
    # TODO: Update field names below to match actual dataset schema
    print(f"QUESTION : {example.get('question', 'N/A')}")
    print(f"ANSWER   : {example.get('answer', 'N/A')}")
    context = example.get('context', example.get('passage', 'N/A'))
    print(f"CONTEXT  : {str(context)[:300]}...")
    if 'raw_ocr' in example:
        print(f"RAW OCR  : {str(example['raw_ocr'])[:300]}...")
    print()

Example 1
QUESTION : Who is the author of the book, "Horrors of Slavery, or the American Turf in Tripoli"?
ANSWER   : WILLIAM RAY
CONTEXT  : Aiscellaneous Repository. From the Albany Register, WAR, OR A PROSPECT OF IT, From recent instances of British Outrage. BY: WILLIAM RAY, Author of the contemplated publication, entitled, “Horrors of Slavery, or the American Turf in Tripoli,” VOTARIES of Freedom, arm! The British Lion roars! Legions ...
RAW OCR  : fAiscellancous Bepogitory.
. dvom the Albany Regifier,
. . WAR,OR A PROSPECT OF IT,
From recent inflances of Britifp Oulrage.
 BY: WILLIAM RAY:
HAuthsr of the tontemplated publication, entitled,
« Horrors of Slavery,or the American Turg
in Tripoli,”
VOT’RIES of Freedom, arm!
 The British Lion roars ...

Example 2
QUESTION : Who was the Grand Officer of the Legion of Honor?
ANSWER   : de Rosemberg
CONTEXT  : Surely he above the rest of his fellow mortals, partakes of heaven here below, of bliss which none but the virtuous ever claim. ¥ Obi

In [39]:
# Field name aliases — update if dataset uses different keys
# TODO: Confirm these field names against actual dataset schema
FIELD_QUESTION = "question"
FIELD_ANSWER   = "answer"
FIELD_CONTEXT  = "context"      # cleaned / gold context
FIELD_RAW_OCR  = "raw_ocr"      # raw OCR text (may not exist)

HAS_RAW_OCR = FIELD_RAW_OCR in train_ds.column_names
print(f"Dataset has raw OCR field: {HAS_RAW_OCR}")

Dataset has raw OCR field: True


In [40]:
# Length statistics
def compute_length_stats(dataset, field, label, n=None):
    """Compute token-approximate length stats for a text field."""
    subset = dataset.select(range(min(n or len(dataset), len(dataset))))
    lengths = [len(str(ex.get(field, "")).split()) for ex in subset]
    return {
        "field": label,
        "mean": np.mean(lengths),
        "median": np.median(lengths),
        "p95": np.percentile(lengths, 95),
        "max": np.max(lengths),
    }

stats_rows = [
    compute_length_stats(train_ds, FIELD_QUESTION, "Question"),
    compute_length_stats(train_ds, FIELD_ANSWER,   "Answer"),
    compute_length_stats(train_ds, FIELD_CONTEXT,  "Context (clean)"),
]
if HAS_RAW_OCR:
    stats_rows.append(compute_length_stats(train_ds, FIELD_RAW_OCR, "Context (OCR)"))

stats_df = pd.DataFrame(stats_rows).set_index("field").round(1)
print("Word-count statistics (train split):")
display(stats_df)

Word-count statistics (train split):


,mean,median,p95,max
field,,,,
Question,11.1,10.0,18.0,30
Answer,2.0,2.0,4.0,47
Context (clean),219.7,225.0,245.0,4077
Context (OCR),233.9,240.0,250.0,1115


---
## 5. Preprocessing

Minimal, reversible preprocessing: choose OCR vs. clean context, normalize whitespace, handle missing values, and truncate long passages.

In [41]:
def normalize_whitespace(text: str) -> str:
    """Collapse multiple spaces/newlines and strip leading/trailing whitespace."""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_context(example: dict, use_raw_ocr: bool = False) -> str:
    """Return the appropriate context field, with fallback."""
    if use_raw_ocr and HAS_RAW_OCR:
        ctx = example.get(FIELD_RAW_OCR, "") or ""
    else:
        ctx = example.get(FIELD_CONTEXT, "") or ""
    return normalize_whitespace(str(ctx))


def truncate_words(text: str, max_words: int) -> str:
    """Hard-truncate text to max_words (word boundary, not token boundary)."""
    words = text.split()
    return " ".join(words[:max_words])


# Approximate word limit for context (generous; tokenizer may cut further)
CONTEXT_MAX_WORDS = int(MAX_INPUT_LENGTH * 0.6)


def preprocess_example(example: dict) -> dict:
    """Apply all preprocessing to a single dataset example."""
    question = normalize_whitespace(str(example.get(FIELD_QUESTION, "") or ""))
    answer   = normalize_whitespace(str(example.get(FIELD_ANSWER,   "") or ""))
    context  = get_context(example, use_raw_ocr=USE_RAW_OCR)
    context  = truncate_words(context, CONTEXT_MAX_WORDS)
    return {
        "question": question,
        "answer":   answer,
        "context":  context,
    }


print("Applying preprocessing ...")
train_clean = train_ds.map(preprocess_example, desc="Preprocess train")
dev_clean   = dev_ds.map(preprocess_example,   desc="Preprocess dev")
test_clean  = test_ds.map(preprocess_example,  desc="Preprocess test")

# Sanity check
ex = train_clean[0]
assert ex["question"], "Question should not be empty after preprocessing"
print(f"Sample after preprocessing:")
print(f"  Q: {ex['question']}")
print(f"  A: {ex['answer']}")
print(f"  Context ({len(ex['context'].split())} words): {ex['context'][:200]}...")

Applying preprocessing ...


Preprocess train:   0%|          | 0/439302 [00:00<?, ? examples/s]

Preprocess dev:   0%|          | 0/24111 [00:00<?, ? examples/s]

Preprocess test:   0%|          | 0/24084 [00:00<?, ? examples/s]

Sample after preprocessing:
  Q: Who is the author of the book, "Horrors of Slavery, or the American Turf in Tripoli"?
  A: WILLIAM RAY
  Context (196 words): Aiscellaneous Repository. From the Albany Register, WAR, OR A PROSPECT OF IT, From recent instances of British Outrage. BY: WILLIAM RAY, Author of the contemplated publication, entitled, “Horrors of S...


---
## 6. Prompt Formatting

We use an instruction-style prompt compatible with Phi-3.5's chat template. The same formatting function is shared between inference and training.

In [43]:
SYSTEM_PROMPT = (
    "You are answering questions about historical American newspapers. "
    "Use the provided context to answer the question briefly and accurately. "
    'If the answer is not supported by the context, say \"Not enough information.\".'
)


def build_user_message(context: str, question: str) -> str:
    """Construct the user turn content (context + question)."""
    return (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )


def format_prompt_for_inference(context: str, question: str) -> str:
    """Return a plain-text prompt for zero-shot / fine-tuned generation."""
    user_msg = build_user_message(context, question)
    # Phi-3.5 uses a chat-style template; we replicate the structure manually
    # so we can also apply it during tokenizer.apply_chat_template if preferred.
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{user_msg}<|end|>\n"
        "<|assistant|>\n"
    )
    return prompt


def format_prompt_for_training(context: str, question: str, answer: str) -> str:
    """Return a complete prompt+answer string used for SFT loss computation."""
    prompt = format_prompt_for_inference(context, question)
    return prompt + answer + "<|end|>"


# Quick sanity check
sample = train_clean[0]
print("=== Inference prompt (first 600 chars) ===")
print(format_prompt_for_inference(sample["context"], sample["question"])[:600])
print("\n=== Training sequence (first 600 chars) ===")
print(format_prompt_for_training(sample["context"], sample["question"], sample["answer"])[:600])

=== Inference prompt (first 600 chars) ===
<|system|>
You are answering questions about historical American newspapers. Use the provided context to answer the question briefly and accurately. If the answer is not supported by the context, say "Not enough information.".<|end|>
<|user|>
Context:
Aiscellaneous Repository. From the Albany Register, WAR, OR A PROSPECT OF IT, From recent instances of British Outrage. BY: WILLIAM RAY, Author of the contemplated publication, entitled, “Horrors of Slavery, or the American Turf in Tripoli,” VOTARIES of Freedom, arm! The British Lion roars! Legions of Valor, take th’ alarm—; Rash, rush to guard o

=== Training sequence (first 600 chars) ===
<|system|>
You are answering questions about historical American newspapers. Use the provided context to answer the question briefly and accurately. If the answer is not supported by the context, say "Not enough information.".<|end|>
<|user|>
Context:
Aiscellaneous Repository. From the Albany Register, WAR, OR